## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import StratifiedKFold

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Methodology
### Objective: 
- Ads at Captain Dispatch - learnings for patience (COBRA)

### Segmentation Criteria: 
- Customer Entry Criteria (Bangalore ,Hyderabad, Delhi, Jaipur, Kolkata <> Link & Auto) 
- Customer Metrics (LTR <> FE <> RR <> NET <> Segment <> COBRA <> COBRM <> OCARA <> ETA <> TTC <> patience_time <> expired)
- Customer Service Affinity (LINK_ONLY <> BOTH <> AUTO_ONLY) 
- Optional Demographics info (Gender)

### Collect Data: 
- fare_estimates_enriched <> order_logs_snapshot/immutable/fact <> iallocator_customer_segments
- Create a customer-level aggregated view.

### Stratified Random Sampling: 
- Use randomization to assign customers within each segment to either the control group (A) or the treatment group (B & C).
- <b>Balance Groups:</b>     
    - Ensure that the size of the groups is balanced. 
    - Conduct a pre-analysis to check if the groups are indeed homogeneous before starting the experiment.
    - This helps in confirming that the segmentation strategy is effective.
- <b>Exclude Outliers:</b>
    - Identify and exclude outliers that might skew the results. 
    - Outliers could be customers with extremely high or low values in the chosen criteria that are not representative of the majority.
	

### Implement the Experiment: 
- Implement the changes for the treatment group and keep the control group unchanged. 
	
### Monitor and Analyse: 
- Monitor the experiment in real-time and, once completed, analyse the results to determine if there are significant differences between the groups.

## Parameter

In [4]:
current_working_directory = os.getcwd()
print(current_working_directory)

/Users/rapido/analytics/customer/Ads-Monetisation/captian_dispatch


In [5]:
local_datasets = '/Users/rapido/local-datasets/customer/ads-monetisation/captian_dispatch_exp/'

In [6]:
## Parameter 
start_date = '20240108'
end_date = '20240121'
cities = ['Bangalore' ,'Hyderabad', 'Delhi', 'Jaipur', 'Kolkata']
services = ['Link', 'Auto']

In [7]:
# Convert the lists to comma-separated strings
city_str = "', '".join(cities)
service_str = "', '".join(services)
print(city_str)
print(service_str)

Bangalore', 'Hyderabad', 'Delhi', 'Jaipur', 'Kolkata
Link', 'Auto


In [8]:
service_mapping_query = f"""
    SELECT 
        city_display_name city,
        service_level service_name,
        service_detail_id
    FROM 
        datasets.service_mapping
    WHERE 
        city_display_name IN ('{city_str}')
        AND service_level IN ('{service_str}')
"""

df_service_mapping = pd.read_sql(service_mapping_query, connection)
df_service_mapping

,city,service_name,service_detail_id
0,Hyderabad,Link,574013f14fdf4798208bba26
1,Jaipur,Link,5ba1f6d0be5b826084cc7202
2,Kolkata,Link,5c124a1e0241913903a42fe1
3,Delhi,Auto,5fc8c40bf4f0a70007d7cfbc
4,Hyderabad,Auto,5ef2bc5b85846b775f97d170
5,Delhi,Link,5da4660028af187d8d52cc3c
6,Bangalore,Auto,5c53562fceb6fc9241980547
7,Bangalore,Link,57370b61a6855d70057417d1
8,Jaipur,Auto,5f6e099d5a5f95000789ead6


In [9]:
service_detail_id_list = df_service_mapping.service_detail_id.tolist()
service_detail_id_str = "', '".join(service_detail_id_list)
print(service_detail_id_list)
print(service_detail_id_str)

['574013f14fdf4798208bba26', '5ba1f6d0be5b826084cc7202', '5c124a1e0241913903a42fe1', '5fc8c40bf4f0a70007d7cfbc', '5ef2bc5b85846b775f97d170', '5da4660028af187d8d52cc3c', '5c53562fceb6fc9241980547', '57370b61a6855d70057417d1', '5f6e099d5a5f95000789ead6']
574013f14fdf4798208bba26', '5ba1f6d0be5b826084cc7202', '5c124a1e0241913903a42fe1', '5fc8c40bf4f0a70007d7cfbc', '5ef2bc5b85846b775f97d170', '5da4660028af187d8d52cc3c', '5c53562fceb6fc9241980547', '57370b61a6855d70057417d1', '5f6e099d5a5f95000789ead6


## Datasets

### Fetching data

In [10]:
service_detail_id_list

['574013f14fdf4798208bba26',
 '5ba1f6d0be5b826084cc7202',
 '5c124a1e0241913903a42fe1',
 '5fc8c40bf4f0a70007d7cfbc',
 '5ef2bc5b85846b775f97d170',
 '5da4660028af187d8d52cc3c',
 '5c53562fceb6fc9241980547',
 '57370b61a6855d70057417d1',
 '5f6e099d5a5f95000789ead6']

##customer_funnel

customer_funnel = f"""

    WITH fe_tbl AS (

        SELECT
            DATE_FORMAT(DATE_PARSE(fee.yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            fee.city,
            fee.service_name,
            user_id as user_id,
            fee.fare_estimate_id,
            COALESCE(price_increase_strategy,'no_surge') AS strategy,
            sfe.session_service_level,
            sfe.session_id
        FROM
            hive.pricing.fare_estimates_enriched fee
        LEFT JOIN hive.marketplace.session_fare_estimates sfe    
            ON sfe.yyyymmdd = fee.yyyymmdd
            AND sfe.fare_estimate_id = fee.fare_estimate_id
            AND sfe.service_name = fee.service_name
        WHERE 
            fee.yyyymmdd BETWEEN '{start_date}' 
            AND '{end_date}'
            AND fee.service_detail_id IN ('{service_detail_id_str}')
    ),

    rr_net_tbl as (

        SELECT   
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            city_name,
            service_obj_service_name service_name,
            service_detail_id,
            customer_id,
            customer_obj_mobile customer_mobile,
            estimate_id fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag,
            lower(modified_order_status) modified_order_status,
            eta,
            CASE 
            WHEN accepted_epoch IS NOT NULL THEN (accepted_epoch/1000 - order_create_epoch/1000)
            ELSE 0 END AS patience_time,
            CASE 
            WHEN accepted_epoch IS NOT NULL THEN (customer_cancelled_epoch/1000 - order_create_epoch/1000)
            ELSE 0 END AS time_to_cancel
        FROM 
            orders.order_logs_fact
        WHERE
            yyyymmdd BETWEEN '{start_date}' 
            AND '{end_date}'
            AND service_detail_id IN ('{service_detail_id_str}')
    ),

    fe_merged AS (
        SELECT
            fe_tbl.*,
            rr_net_tbl.customer_id,
            rr_net_tbl.order_id,
            rr_net_tbl.customer_mobile,
            rr_net_tbl.order_status,
            rr_net_tbl.spd_fraud_flag,
            rr_net_tbl.modified_order_status,
            COALESCE(rr_net_tbl.time_to_cancel,0) time_to_cancel,
            COALESCE(rr_net_tbl.eta,0) eta,
            COALESCE(rr_net_tbl.patience_time, 0) patience_time
        FROM
            fe_tbl
        LEFT JOIN
            rr_net_tbl
            ON fe_tbl.order_date = rr_net_tbl.order_date
            AND fe_tbl.user_id = rr_net_tbl.customer_id
            AND fe_tbl.fare_estimate_id = rr_net_tbl.fare_estimate_id
            AND fe_tbl.service_name = rr_net_tbl.service_name
    )

    SELECT 
        *
    FROM
    (
    SELECT
        user_id,

        MIN(customer_mobile) customer_mobile,

        COUNT(distinct fare_estimate_id) AS fe_count,
        COUNT(distinct order_id) AS rr_count,
        COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count,

        COUNT(DISTINCT session_id) fe_session,
        COUNT(DISTINCT CASE WHEN order_id IS NOT NULL THEN session_id END) gross_sessions,
        COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN session_id END) net_sessions,

        COUNT(DISTINCT CASE WHEN modified_order_status = 'cobrm' THEN order_id END) cobrm,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'cobra' THEN order_id END) cobra,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'ocara' THEN order_id END) ocara,
        COUNT(DISTINCT CASE WHEN modified_order_status = 'expired' THEN order_id END) expired,
        SUM(time_to_cancel) time_to_cancel,
        SUM(eta) eta,
        SUM(patience_time) patience_time

    FROM 
        fe_merged
    GROUP BY 1
    )
    WHERE rr_count > 0
"""

df_customer_funnel = pd.read_sql(customer_funnel, connection)
df_customer_funnel.to_csv(local_datasets + 'raw/customer_funnel_{}_to_{}.csv'\
                          .format(start_date, end_date),
                          index=False)

##customer_segment

customer_segment = f"""

    WITH fe_tbl AS (
    
        SELECT
            customer_id user_id,
            city_name
        FROM
        (
        SELECT   
            customer_id,
            city_name,
            row_number() over(partition by customer_id order by epoch desc) seq_no
        FROM 
            orders.order_logs_fact
        WHERE
            yyyymmdd BETWEEN '{start_date}' 
            AND '{end_date}'
            AND service_detail_id IN ('{service_detail_id_str}')
        )
        WHERE seq_no = 1
    )

        SELECT
            user_id,
            COALESCE(city_name, 'UNKNOWN') city,
            COALESCE(gender, 'UNKNOWN') gender,
            ltr_jan1,
            ltr_jan21,
            jan_net_rides,
            COALESCE(lifetime_stage_jan1, 'NA') lifetime_stage_jan1,
            COALESCE(lifetime_stage_jan21, 'NA') lifetime_stage_jan21,
            COALESCE(recency_segment_jan1, 'NA') recency_segment_jan1,
            COALESCE(recency_segment_jan21, 'NA') recency_segment_jan21,
            COALESCE(service_affinity_jan1, 'NA') service_affinity_jan1,
            COALESCE(service_affinity_jan21, 'NA') service_affinity_jan21
        FROM
        (    
            SELECT 
                user_id,
                city_name,
                MIN(gender_tag) gender,
                MIN(taxi_lifetime_rides) ltr_jan1,
                MAX(taxi_lifetime_rides) ltr_jan21,
                MAX(taxi_lifetime_rides) - MIN(taxi_lifetime_rides) jan_net_rides, 
                MAX(CASE WHEN run_date = '2024-01-01' THEN taxi_lifetime_stage END) lifetime_stage_jan1,
                MAX(CASE WHEN run_date = '2024-01-21' THEN taxi_lifetime_stage END) lifetime_stage_jan21,
                MAX(CASE WHEN run_date = '2024-01-01' THEN taxi_recency_segment END) recency_segment_jan1,
                MAX(CASE WHEN run_date = '2024-01-21' THEN taxi_recency_segment END) recency_segment_jan21,
                MAX(CASE WHEN run_date = '2024-01-01' THEN taxi_service_affinity END) service_affinity_jan1,
                MAX(CASE WHEN run_date = '2024-01-21' THEN taxi_service_affinity END) service_affinity_jan21
            FROM 
                fe_tbl
            LEFT JOIN 
                datasets.iallocator_customer_segments
                ON user_id = customer_id 
                AND (run_date = '2024-01-01' OR run_date = '2024-01-21') 
            GROUP BY 1,2
        )

"""

df_customer_segment = pd.read_sql(customer_segment, connection)

##Local copy
df_customer_segment.to_csv(local_datasets + 'raw/customer_segment_{}_to_{}.csv' 
                           .format(start_date, end_date), index=False 
                          )

### Reading local extract

In [92]:
df_customer_funnel = pd.read_csv(local_datasets + 'raw/customer_funnel_{}_to_{}.csv'
                                 .format(start_date, end_date)
                                )
df_customer_segment = pd.read_csv(local_datasets + 'raw/customer_segment_{}_to_{}.csv'
                                 .format(start_date, end_date)
                                )

In [93]:
print(df_customer_funnel.shape)
print(df_customer_funnel.user_id.nunique())
print(df_customer_segment.shape)
print(df_customer_segment.user_id.nunique())

(4420480, 15)
4420480
(4436711, 12)
4436711


In [94]:
df_customer_funnel.head(2)

,user_id,customer_mobile,fe_count,rr_count,net_count,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,time_to_cancel,eta,patience_time
0,642d5b6d1ecc3abb0aaa8c2c,8.977909e+09,24,5,2,13,3,2,0,3,0,0,0,33.0,29
1,658bc1f51827f9451107d0ea,9.911560e+09,14,6,2,5,4,2,0,0,4,0,1873,22.0,744


In [ ]:
df_customer_funnel['customer_mobile'] = df_customer_funnel['customer_mobile'].astype(float).astype(int)

In [ ]:
df_customer_segment.head(2)

In [ ]:
## Joining DataFrames
df_customer_merge_raw = pd.merge(df_customer_funnel, df_customer_segment, on='user_id')

df_customer_merge = df_customer_merge_raw.copy()
df_customer_merge.head(5)

In [ ]:
df_customer_merge['fe2rr'] = df_customer_merge['rr_count']*100.0/df_customer_merge['fe_count']
df_customer_merge['fe2net'] = df_customer_merge['net_count']*100.0/df_customer_merge['fe_count']
df_customer_merge['g2n'] = df_customer_merge['net_count']*100.0/df_customer_merge['rr_count']
df_customer_merge['g2n_session'] = df_customer_merge['net_sessions']*100.0/df_customer_merge['gross_sessions']
df_customer_merge['fe2rr_session'] = df_customer_merge['gross_sessions']*100.0/df_customer_merge['fe_session']
df_customer_merge['fe2net_session'] = df_customer_merge['net_sessions']*100.0/df_customer_merge['fe_session']
df_customer_merge = df_customer_merge.round(2)

In [ ]:
df_customer_merge = df_customer_merge[['user_id', 'city', 'gender', 'customer_mobile',
                                       'fe_count', 'rr_count', 'net_count', 
                                       'fe2rr', 'g2n', 'fe2net',
                                       'fe_session','gross_sessions', 'net_sessions',
                                       'cobrm', 'cobra', 'ocara', 'expired', 'eta', 'patience_time',
                                       'time_to_cancel',
                                       'fe2rr_session', 'g2n_session', 'fe2net_session',
                                       'ltr_jan1', 'ltr_jan21', 'jan_net_rides',
                                       'lifetime_stage_jan1', 'lifetime_stage_jan21',
                                       'recency_segment_jan1', 'recency_segment_jan21',
                                       'service_affinity_jan1', 'service_affinity_jan21'
                                      ]]

In [ ]:
df_customer_merge

In [ ]:
df_customer_merge[ df_customer_merge['user_id'].isin(['631216f8f9a036dc3c412ac4'])].T

#### Other EDA

In [ ]:
# print(df_customer_merge.user_id.nunique(), ' - 21days active(fe) customers')
print(df_customer_merge[df_customer_merge['rr_count'] > 0].user_id.nunique(), ' - 21days rr customers')
print(df_customer_merge[df_customer_merge['net_count'] > 0].user_id.nunique(), ' - 21days net customers')

In [ ]:
## Data Filter 1 

df_customer_merge \
[df_customer_merge['rr_count'] > 0].user_id.nunique()

In [ ]:
df_stratified_random_sampling = df_customer_merge \
                                    [df_customer_merge['rr_count'] > 0]
df_stratified_random_sampling.head(5)

In [ ]:
print(df_stratified_random_sampling.shape)
print(df_stratified_random_sampling.user_id.nunique())

In [ ]:
df_stratified_random_sampling.city.unique()

In [ ]:
df_stratified_random_sampling.groupby(['city']).user_id.nunique()

In [ ]:
df_stratified_random_sampling = df_stratified_random_sampling[df_stratified_random_sampling['city']\
                                                              .isin(['Hyderabad', 'Delhi', 'Jaipur', 
                                                                     'Bangalore', 'Kolkata'])]

In [ ]:
df_stratified_random_sampling['customer_mobile'] = df_stratified_random_sampling['customer_mobile'].apply(str)

In [ ]:
df_stratified_random_sampling.head(5)

In [ ]:
df_stratified_random_sampling[['city', 'gender',
                               'fe_session','gross_sessions', 'net_sessions',
                               'cobrm', 'cobra', 'ocara', 'expired', 'eta', 'patience_time', 'time_to_cancel',
                               'fe2rr_session', 'g2n_session', 'fe2net_session',
                               'lifetime_stage_jan21','lifetime_stage_jan21',
                               'recency_segment_jan1', 'recency_segment_jan21',
                               'service_affinity_jan1', 'service_affinity_jan21'
                              ]].isnull().sum()

In [ ]:
# Columns to fill null values in
columns_to_fill = ['lifetime_stage_jan1', 'lifetime_stage_jan21',
                   'recency_segment_jan1', 'recency_segment_jan21', 
                   'service_affinity_jan1', 'service_affinity_jan21']

# Fill null values in the specified columns with a specific value (e.g., 0)
df_stratified_random_sampling[columns_to_fill] = df_stratified_random_sampling[columns_to_fill].fillna('NA')

In [ ]:
df_stratified_random_sampling[['city', 'gender',
                               'fe_session','gross_sessions', 'net_sessions',
                               'cobrm', 'cobra', 'ocara', 'expired', 'eta', 'patience_time',
                               'fe2rr_session', 'g2n_session', 'fe2net_session',
                               'lifetime_stage_jan21','lifetime_stage_jan21',
                               'recency_segment_jan1', 'recency_segment_jan21',
                               'service_affinity_jan1', 'service_affinity_jan21'
                              ]].isnull().sum()

In [ ]:
df_stratified_random_sampling = df_stratified_random_sampling[df_stratified_random_sampling['g2n_session'].notna()]

In [ ]:
df_stratified_random_sampling[['city', 'gender',
                               'fe_session','gross_sessions', 'net_sessions',
                               'cobrm', 'cobra', 'ocara', 'expired', 'eta', 'patience_time',
                               'fe2rr_session', 'g2n_session', 'fe2net_session',
                               'lifetime_stage_jan21','lifetime_stage_jan21',
                               'recency_segment_jan1', 'recency_segment_jan21',
                               'service_affinity_jan1', 'service_affinity_jan21'
                              ]].isnull().sum()

In [ ]:
df_stratified_random_sampling.info()

In [ ]:
df_customer_merge[ df_customer_merge['user_id'].isin(['631216f8f9a036dc3c412ac4'])].T

## Stratified Random Sampling

In [136]:
df_stratified_random_sampling.shape

(4420045, 32)

In [137]:
df_stratified_random_sampling.user_id.nunique()

4420045

In [181]:
df_stratified_random_sampling['customer_mobile'] = pd.to_numeric(df_stratified_random_sampling['customer_mobile'], errors='coerce').astype(float)

# Drop rows with NaN values in any column
df_stratified_random_sampling_cleaned = df_stratified_random_sampling.dropna()

In [182]:
df_stratified_random_sampling_cleaned.shape

(4419876, 32)

In [183]:
df_stratified_random_sampling_cleaned.user_id.nunique()

4419876

In [ ]:
stratify_grouper_col =['fe_session','gross_sessions', 'net_sessions',
                       'cobrm', 'cobra', 'ocara', 'expired', 'eta', 'patience_time'
                      ]

In [184]:
random_seed = 42

# Split the df into two parts for initial 'control' and 'temp'
df_train_control, df_temp = train_test_split(df_stratified_random_sampling_cleaned, 
                                             test_size=1/3, 
                                             random_state=random_seed
#                                              stratify=df_stratified_random_sampling_cleaned[stratify_grouper_col]
                                            )

# Further split 'temp' into 'test1' and 'test2'
df_test1, df_test2 = train_test_split(df_temp, 
                                      test_size=0.5, 
                                      random_state=random_seed
#                                       stratify=df_temp[stratify_grouper_col]
                                     )

df_train_control = df_train_control.sample(min(300000, len(df_train_control)), random_state=random_seed)
df_test1 = df_test1.sample(min(300000, len(df_test1)), random_state=random_seed)
df_test2 = df_test2.sample(min(300000, len(df_test2)), random_state=random_seed)

# Assign group labels
df_train_control['group_tc'] = 'control'
df_test1['group_tc'] = 'test1'
df_test2['group_tc'] = 'test2'

df_final_sample = pd.concat([df_train_control, df_test1, df_test2])

In [185]:
print(df_stratified_random_sampling_cleaned.shape)
print(df_final_sample.shape)

(4419876, 32)
(900000, 33)


In [186]:
df_final_sample

,user_id,city,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,time_to_cancel,fe2rr_session,g2n_session,fe2net_session,ltr_jan1,ltr_jan21,jan_net_rides,lifetime_stage_jan1,lifetime_stage_jan21,recency_segment_jan1,recency_segment_jan21,service_affinity_jan1,service_affinity_jan21,group_tc
685027,5eddb5869adf085bf4f6a237,Hyderabad,MALE,9.908611e+09,30,3,2,10.00,66.67,6.67,7,2,2,0,0,0,1,6.0,73,0,28.57,100.0,28.57,10.0,12.0,2.0,HOOK,HOOK,RECENT,STATIONARY,ONLY_LINK,ONLY_LINK,control
864266,59a3f951f2b6708059499182,Bangalore,MALE,9.964336e+09,7,4,3,57.14,75.00,42.86,4,3,3,0,1,0,0,51.0,2240,0,75.00,100.0,75.00,67.0,74.0,7.0,COMMITTED,CHURN_OTB,RECENT,STATIONARY,ONLY_LINK,ONLY_LINK,control
1962366,62ea23dbcc3711c71c9144d2,Bangalore,FEMALE,6.364831e+09,4,1,1,25.00,100.00,25.00,2,1,1,0,0,0,0,20.0,12,0,50.00,100.0,50.00,0.0,1.0,1.0,UNKNOWN,HANDHOLDING,UNKNOWN,STATIONARY,UNKNOWN,ONLY_LINK,control
2592507,5cc68d861acc4f1c5503e8f4,Hyderabad,FEMALE,9.032534e+09,2,1,0,50.00,0.00,0.00,1,1,0,0,1,0,0,0.0,0,0,100.00,0.0,0.00,78.0,78.0,0.0,SOFT_CHURN,DORMANT,STATIONARY,DORMANT,BOTH,ONLY_LINK,control
3984576,627e277adc46f329781d3ae7,Bangalore,MALE,7.760409e+09,3,1,0,33.33,0.00,0.00,3,1,0,0,0,1,0,1.0,459,1030,33.33,0.0,0.00,4.0,4.0,0.0,INACTIVE,INACTIVE,INACTIVE,INACTIVE,UNKNOWN,ONLY_AUTO,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2948862,5db724bec54ee1212528e56d,Hyderabad,MALE,7.730004e+09,3,2,2,66.67,100.00,66.67,3,2,2,0,0,0,0,36.0,272,0,66.67,100.0,66.67,547.0,549.0,2.0,DORMANT,SUSTENANCE,DORMANT,STATIONARY,ONLY_LINK,ONLY_LINK,test2
2819954,5e252a7b48bf6e0289861ef2,Delhi,MALE,7.906961e+09,10,3,1,30.00,33.33,10.00,4,1,1,1,0,0,1,26.0,24,0,25.00,100.0,25.00,9.0,10.0,1.0,DORMANT,HOOK,DORMANT,RECENT,ONLY_LINK,ONLY_LINK,test2
1480452,62ef8b7faa737791851e006b,Jaipur,FEMALE,9.100134e+09,7,1,0,14.29,0.00,0.00,4,1,0,0,0,1,0,6.0,73,135,25.00,0.0,0.00,25.0,25.0,0.0,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_AUTO,ONLY_AUTO,test2
2930040,647ab29e0b7f5804c18392ea,Bangalore,MALE,8.147345e+09,8,3,0,37.50,0.00,0.00,3,1,0,0,0,3,0,11.0,32,1497,33.33,0.0,0.00,18.0,18.0,0.0,HOOK,DORMANT,STATIONARY,DORMANT,UNKNOWN,UNKNOWN,test2


In [187]:
df_final_sample['customer_mobile'].nunique()

900000

### Balance Groups

In [188]:
df_final_sample.groupby(['group_tc'])\
.agg(user_base = pd.NamedAgg('user_id', 'nunique'),
     customer_mobile = pd.NamedAgg('customer_mobile', 'nunique'),
     city_distr = pd.NamedAgg(column='city', 
                              aggfunc=lambda x: x.value_counts().to_dict()),
     fe_count = pd.NamedAgg('fe_count', 'sum'),
     rr_count = pd.NamedAgg('rr_count', 'sum'),
     net_count = pd.NamedAgg('net_count', 'sum'),
     fe_session = pd.NamedAgg('fe_session', 'sum'),
     gross_sessions = pd.NamedAgg('gross_sessions', 'sum'),
     net_sessions = pd.NamedAgg('net_sessions', 'sum'),
     cobrm = pd.NamedAgg('cobrm', 'sum'),
     cobra = pd.NamedAgg('cobra', 'sum'),
     ocara = pd.NamedAgg('ocara', 'sum'),
     expired = pd.NamedAgg('expired', 'sum'),
     eta = pd.NamedAgg('eta', 'sum'),
     patience_time = pd.NamedAgg('patience_time', 'sum'),
     lifetime_stage_dist = pd.NamedAgg(column='lifetime_stage_jan21', 
                                        aggfunc=lambda x: x.value_counts().to_dict()),   
     recency_segment_dist = pd.NamedAgg(column='recency_segment_jan21', 
                                        aggfunc=lambda x: x.value_counts().to_dict()),     
     service_affinity_dist = pd.NamedAgg(column='service_affinity_jan21', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),
     gender_distr = pd.NamedAgg(column='gender', 
                                aggfunc=lambda x: x.value_counts().to_dict()),
     median_ltr_jan1 = pd.NamedAgg('ltr_jan1', 'median'),
     median_ltr_jan21 = pd.NamedAgg('ltr_jan21', 'median')     
    ).T

group_tc,control,test1,test2
user_base,300000,300000,300000
customer_mobile,300000,300000,300000
city_distr,"{'Hyderabad': 120295, 'Bangalore': 83017, 'Delhi': 62186, 'Jaipur': 17984, 'Kolkata': 16518}","{'Hyderabad': 120326, 'Bangalore': 83397, 'Delhi': 61545, 'Jaipur': 18155, 'Kolkata': 16577}","{'Hyderabad': 120569, 'Bangalore': 82947, 'Delhi': 62067, 'Jaipur': 17900, 'Kolkata': 16517}"
fe_count,3027927,3022514,3033076
rr_count,1097639,1094194,1094472
net_count,604884,602813,602269
fe_session,1278540,1276326,1278957
gross_sessions,854332,851176,852067
net_sessions,604797,602722,602187
cobrm,9979,10314,10067


In [189]:
df_final_sample_city = df_final_sample.groupby(['city', 'group_tc'])\
.agg(user_base = pd.NamedAgg('user_id', 'nunique'),
     fe_count = pd.NamedAgg('fe_count', 'sum'),
     rr_count = pd.NamedAgg('rr_count', 'sum'),
     net_count = pd.NamedAgg('net_count', 'sum'),
     fe_session = pd.NamedAgg('fe_session', 'sum'),
     gross_sessions = pd.NamedAgg('gross_sessions', 'sum'),
     net_sessions = pd.NamedAgg('net_sessions', 'sum'),
     cobrm = pd.NamedAgg('cobrm', 'sum'),
     cobra = pd.NamedAgg('cobra', 'sum'),
     ocara = pd.NamedAgg('ocara', 'sum'),
     expired = pd.NamedAgg('expired', 'sum'),
     eta = pd.NamedAgg('eta', 'sum'),
     patience_time = pd.NamedAgg('patience_time', 'sum'),
     lifetime_stage_dist = pd.NamedAgg(column='lifetime_stage_jan21', 
                                        aggfunc=lambda x: x.value_counts().to_dict()),  
     recency_segment_dist = pd.NamedAgg(column='recency_segment_jan21', 
                                        aggfunc=lambda x: x.value_counts().to_dict()),     
     service_affinity_dist = pd.NamedAgg(column='service_affinity_jan21', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),
     gender_distr = pd.NamedAgg(column='gender', 
                                aggfunc=lambda x: x.value_counts().to_dict()),
     median_ltr_jan1 = pd.NamedAgg('ltr_jan1', 'median'),
     median_ltr_jan21 = pd.NamedAgg('ltr_jan21', 'median')     
    ).reset_index()
df_final_sample_city

,city,group_tc,user_base,fe_count,rr_count,net_count,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,lifetime_stage_dist,recency_segment_dist,service_affinity_dist,gender_distr,median_ltr_jan1,median_ltr_jan21
0,Bangalore,control,83017,893798,355739,186291,373214,266094,186260,1638,72620,64281,30210,2963628.0,39846196,"{'HOOK': 24791, 'COMMITTED': 15766, 'HANDHOLDING': 12128, 'SUSTENANCE': 10611, 'DORMANT': 7676, 'CHURN_OTB': 3959, 'INACTIVE': 3540, 'UNKNOWN': 2301, 'SOFT_CHURN': 1871, 'DETOX': 374}","{'RECENT': 43364, 'STATIONARY': 26136, 'DORMANT': 7676, 'INACTIVE': 3540, 'UNKNOWN': 2301}","{'ONLY_AUTO': 36911, 'ONLY_LINK': 29585, 'BOTH': 12255, 'UNKNOWN': 4266}","{'MALE': 56125, 'FEMALE': 24305, 'UNKNOWN': 2331, 'OTHERS': 256}",16.0,19.0
1,Bangalore,test1,83397,897205,355252,186602,375607,266357,186571,1598,72699,63299,30341,2967630.0,39492055,"{'HOOK': 24871, 'COMMITTED': 15863, 'HANDHOLDING': 12347, 'SUSTENANCE': 10778, 'DORMANT': 7672, 'CHURN_OTB': 3821, 'INACTIVE': 3514, 'UNKNOWN': 2369, 'SOFT_CHURN': 1876, 'DETOX': 286}","{'RECENT': 43466, 'STATIONARY': 26376, 'DORMANT': 7672, 'INACTIVE': 3514, 'UNKNOWN': 2369}","{'ONLY_AUTO': 36911, 'ONLY_LINK': 29569, 'BOTH': 12513, 'UNKNOWN': 4404}","{'MALE': 56382, 'FEMALE': 24480, 'UNKNOWN': 2303, 'OTHERS': 232}",16.0,19.0
2,Bangalore,test2,82947,891498,352683,184566,372480,264620,184540,1666,71452,64040,30297,2931940.0,39589171,"{'HOOK': 24690, 'COMMITTED': 15817, 'HANDHOLDING': 12228, 'SUSTENANCE': 10706, 'DORMANT': 7569, 'CHURN_OTB': 3767, 'INACTIVE': 3581, 'UNKNOWN': 2433, 'SOFT_CHURN': 1852, 'DETOX': 304}","{'RECENT': 43214, 'STATIONARY': 26150, 'DORMANT': 7569, 'INACTIVE': 3581, 'UNKNOWN': 2433}","{'ONLY_AUTO': 37061, 'ONLY_LINK': 28949, 'BOTH': 12469, 'UNKNOWN': 4468}","{'MALE': 56115, 'FEMALE': 24315, 'UNKNOWN': 2259, 'OTHERS': 258}",16.0,19.0
3,Delhi,control,62186,756929,232391,123066,297903,179051,123051,4965,36589,48731,18605,1814570.0,28695757,"{'HOOK': 14969, 'HANDHOLDING': 12159, 'COMMITTED': 10148, 'DORMANT': 6296, 'SUSTENANCE': 6242, 'UNKNOWN': 3480, 'INACTIVE': 3094, 'CHURN_OTB': 2899, 'DETOX': 1668, 'SOFT_CHURN': 1231}","{'RECENT': 30707, 'STATIONARY': 18609, 'DORMANT': 6296, 'UNKNOWN': 3480, 'INACTIVE': 3094}","{'ONLY_LINK': 37604, 'ONLY_AUTO': 12733, 'BOTH': 6991, 'UNKNOWN': 4858}","{'MALE': 47045, 'FEMALE': 13471, 'UNKNOWN': 1505, 'OTHERS': 165}",11.0,13.0
4,Delhi,test1,61545,739310,226999,119891,290391,174488,119871,5084,35481,47737,18376,1761162.0,27761206,"{'HOOK': 14882, 'HANDHOLDING': 12259, 'COMMITTED': 10088, 'DORMANT': 6246, 'SUSTENANCE': 5910, 'UNKNOWN': 3353, 'INACTIVE': 3075, 'CHURN_OTB': 2849, 'DETOX': 1628, 'SOFT_CHURN': 1255}","{'RECENT': 30392, 'STATIONARY': 18479, 'DORMANT': 6246, 'UNKNOWN': 3353, 'INACTIVE': 3075}","{'ONLY_LINK': 37104, 'ONLY_AUTO': 12683, 'BOTH': 7024, 'UNKNOWN': 4734}","{'MALE': 46502, 'FEMALE': 13417, 'UNKNOWN': 1467, 'OTHERS': 159}",11.0,13.0
5,Delhi,test2,62067,757108,231321,121848,296318,177491,121824,4875,36462,48982,18699,1803017.0,28360560,"{'HOOK': 14799, 'HANDHOLDING': 12254, 'COMMITTED': 10234, 'DORMANT': 6266, 'SUSTENANCE': 6140, 'UNKNOWN': 3324, 'INACTIVE': 3092, 'CHURN_OTB': 2976, 'DETOX': 1640, 'SOFT_CHURN': 1342}","{'RECENT': 30696, 'STATIONARY': 18689, 'DORMANT': 6266, 'UNKNOWN': 3324, 'INACTIVE': 3092}","{'ONLY_LINK': 37441, 'ONLY_AUTO': 12841, 'BOTH': 7104, 'UNKNOWN': 4681}","{'MALE': 46853, 'FEMALE': 13539, 'UNKNOWN': 1522, 'OTHERS': 153}",11.0,13.0
6,Hyderabad,control,120295,1089323,402079,239940,485845,326061,239909,2554,63282,75494,19851,3473915.0,58967886,"{'HOOK': 38241, 'COMMITTED': 22996, 'SUSTENANCE': 15597, 'HANDHOLDING': 15371, 'DORMANT': 10904, 'CHURN_OTB': 6571, 'INACTIVE': 4221, 'SOFT_CHURN': 3108, 'UNKNOWN': 2531, 'DETOX': 755}","{'RECENT': 60260, 'STATIONARY': 42379, 'DORMANT': 10904, 'INACTIVE': 4221, 'UNKNOWN': 2531}","{'ONLY_LINK': 49694, 'ONLY_AUTO': 45958, 'BOTH': 19278, 'UNKNOWN': 5365}","{'MALE': 83895, 'FEMALE': 32769, 'UNKNOWN': 3219, 'OTHERS': 412

In [190]:
df_final_sample_city.columns

Index(['city', 'group_tc', 'user_base', 'fe_count', 'rr_count', 'net_count',
       'fe_session', 'gross_sessions', 'net_sessions', 'cobrm', 'cobra',
       'ocara', 'expired', 'eta', 'patience_time', 'lifetime_stage_dist',
       'recency_segment_dist', 'service_affinity_dist', 'gender_distr',
       'median_ltr_jan1', 'median_ltr_jan21'],
      dtype='object')

In [191]:
df_final_sample_city['fe2rr'] = df_final_sample_city['rr_count']*100.0/df_final_sample_city['fe_count']
df_final_sample_city['fe2net'] = df_final_sample_city['net_count']*100.0/df_final_sample_city['fe_count']
df_final_sample_city['g2n'] = df_final_sample_city['net_count']*100.0/df_final_sample_city['rr_count']
df_final_sample_city['g2n_session'] = df_final_sample_city['net_sessions']*100.0/df_final_sample_city['gross_sessions']
df_final_sample_city['fe2rr_session'] = df_final_sample_city['gross_sessions']*100.0/df_final_sample_city['fe_session']
df_final_sample_city['fe2net_session'] = df_final_sample_city['net_sessions']*100.0/df_final_sample_city['fe_session']

In [192]:
df_final_sample_city

,city,group_tc,user_base,fe_count,rr_count,net_count,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,lifetime_stage_dist,recency_segment_dist,service_affinity_dist,gender_distr,median_ltr_jan1,median_ltr_jan21,fe2rr,fe2net,g2n,g2n_session,fe2rr_session,fe2net_session
0,Bangalore,control,83017,893798,355739,186291,373214,266094,186260,1638,72620,64281,30210,2963628.0,39846196,"{'HOOK': 24791, 'COMMITTED': 15766, 'HANDHOLDING': 12128, 'SUSTENANCE': 10611, 'DORMANT': 7676, 'CHURN_OTB': 3959, 'INACTIVE': 3540, 'UNKNOWN': 2301, 'SOFT_CHURN': 1871, 'DETOX': 374}","{'RECENT': 43364, 'STATIONARY': 26136, 'DORMANT': 7676, 'INACTIVE': 3540, 'UNKNOWN': 2301}","{'ONLY_AUTO': 36911, 'ONLY_LINK': 29585, 'BOTH': 12255, 'UNKNOWN': 4266}","{'MALE': 56125, 'FEMALE': 24305, 'UNKNOWN': 2331, 'OTHERS': 256}",16.0,19.0,39.800827,20.842629,52.367325,69.997820,71.297968,49.907024
1,Bangalore,test1,83397,897205,355252,186602,375607,266357,186571,1598,72699,63299,30341,2967630.0,39492055,"{'HOOK': 24871, 'COMMITTED': 15863, 'HANDHOLDING': 12347, 'SUSTENANCE': 10778, 'DORMANT': 7672, 'CHURN_OTB': 3821, 'INACTIVE': 3514, 'UNKNOWN': 2369, 'SOFT_CHURN': 1876, 'DETOX': 286}","{'RECENT': 43466, 'STATIONARY': 26376, 'DORMANT': 7672, 'INACTIVE': 3514, 'UNKNOWN': 2369}","{'ONLY_AUTO': 36911, 'ONLY_LINK': 29569, 'BOTH': 12513, 'UNKNOWN': 4404}","{'MALE': 56382, 'FEMALE': 24480, 'UNKNOWN': 2303, 'OTHERS': 232}",16.0,19.0,39.595410,20.798145,52.526657,70.045465,70.913748,49.671864
2,Bangalore,test2,82947,891498,352683,184566,372480,264620,184540,1666,71452,64040,30297,2931940.0,39589171,"{'HOOK': 24690, 'COMMITTED': 15817, 'HANDHOLDING': 12228, 'SUSTENANCE': 10706, 'DORMANT': 7569, 'CHURN_OTB': 3767, 'INACTIVE': 3581, 'UNKNOWN': 2433, 'SOFT_CHURN': 1852, 'DETOX': 304}","{'RECENT': 43214, 'STATIONARY': 26150, 'DORMANT': 7569, 'INACTIVE': 3581, 'UNKNOWN': 2433}","{'ONLY_AUTO': 37061, 'ONLY_LINK': 28949, 'BOTH': 12469, 'UNKNOWN': 4468}","{'MALE': 56115, 'FEMALE': 24315, 'UNKNOWN': 2259, 'OTHERS': 258}",16.0,19.0,39.560717,20.702907,52.331981,69.737737,71.042741,49.543600
3,Delhi,control,62186,756929,232391,123066,297903,179051,123051,4965,36589,48731,18605,1814570.0,28695757,"{'HOOK': 14969, 'HANDHOLDING': 12159, 'COMMITTED': 10148, 'DORMANT': 6296, 'SUSTENANCE': 6242, 'UNKNOWN': 3480, 'INACTIVE': 3094, 'CHURN_OTB': 2899, 'DETOX': 1668, 'SOFT_CHURN': 1231}","{'RECENT': 30707, 'STATIONARY': 18609, 'DORMANT': 6296, 'UNKNOWN': 3480, 'INACTIVE': 3094}","{'ONLY_LINK': 37604, 'ONLY_AUTO': 12733, 'BOTH': 6991, 'UNKNOWN': 4858}","{'MALE': 47045, 'FEMALE': 13471, 'UNKNOWN': 1505, 'OTHERS': 165}",11.0,13.0,30.701823,16.258592,52.956440,68.723995,60.103792,41.305727
4,Delhi,test1,61545,739310,226999,119891,290391,174488,119871,5084,35481,47737,18376,1761162.0,27761206,"{'HOOK': 14882, 'HANDHOLDING': 12259, 'COMMITTED': 10088, 'DORMANT': 6246, 'SUSTENANCE': 5910, 'UNKNOWN': 3353, 'INACTIVE': 3075, 'CHURN_OTB': 2849, 'DETOX': 1628, 'SOFT_CHURN': 1255}","{'RECENT': 30392, 'STATIONARY': 18479, 'DORMANT': 6246, 'UNKNOWN': 3353, 'INACTIVE': 3075}","{'ONLY_LINK': 37104, 'ONLY_AUTO': 12683, 'BOTH': 7024, 'UNKNOWN': 4734}","{'MALE': 46502, 'FEMALE': 13417, 'UNKNOWN': 1467, 'OTHERS': 159}",11.0,13.0,30.704170,16.216607,52.815651,68.698707,60.087262,41.279172
5,Delhi,test2,62067,757108,231321,121848,296318,177491,121824,4875,36462,48982,18699,1803017.0,28360560,"{'HOOK': 14799, 'HANDHOLDING': 12254, 'COMMITTED': 10234, 'DORMANT': 6266, 'SUSTENANCE': 6140, 'UNKNOWN': 3324, 'INACTIVE': 3092, 'CHURN_OTB': 2976, 'DETOX': 1640, 'SOFT_CHURN': 1342}","{'RECENT': 30696, 'STATIONARY': 18689, 'DORMANT': 6266, 'UNKNOWN': 3324, 'INACTIVE': 3092}","{'ONLY_LINK': 37441, 'ONLY_AUTO': 12841, 'BOTH': 7104, 'UNKNOWN': 4681}","{'MALE': 46853, 'FEMALE': 13539, 'UNKNOWN': 1522, 'OTHERS': 153}",11.0,13.0,30.553237,16.093873,52.674854,68.636720,59.898825,41.112589
6,Hyderabad,control,120295,1089323,402079,239940,485845,326061,239909,2554,63282,75494,19851,3473915.0,58967886,"{

In [193]:
##Local extract

df_final_sample.to_csv(local_datasets + 'processed/captian_dispatch_exp_group_tc.csv', 
                       index=False )

In [194]:
# Check for common values in 'user_id' among the three DataFrames
common_values_df1_df2 = df_train_control[df_train_control['user_id'].isin(df_test1['user_id'])]['user_id'].tolist()
common_values_df1_df3 = df_train_control[df_train_control['user_id'].isin(df_test2['user_id'])]['user_id'].tolist()

# Find the intersection of common values
common_values_all = set(common_values_df1_df2) & set(common_values_df1_df3)

# Check if common values were found
if common_values_all:
    print(f"Common values in user_id: {list(common_values_all)}")
else:
    print("No common values found.")

No common values found.


In [195]:
print(df_train_control.shape)
print(df_test1.shape)
print(df_test2.shape)
print(df_train_control.user_id.nunique())
print(df_test1.user_id.nunique())
print(df_test2.user_id.nunique())

(300000, 33)
(300000, 33)
(300000, 33)
300000
300000
300000


In [196]:
df_train_control[df_train_control['user_id'].isin(df_test1['user_id'])]

,user_id,city,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,time_to_cancel,fe2rr_session,g2n_session,fe2net_session,ltr_jan1,ltr_jan21,jan_net_rides,lifetime_stage_jan1,lifetime_stage_jan21,recency_segment_jan1,recency_segment_jan21,service_affinity_jan1,service_affinity_jan21,group_tc


In [197]:
df_train_control[df_train_control['user_id'].isin(df_test2['user_id'])]

,user_id,city,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,time_to_cancel,fe2rr_session,g2n_session,fe2net_session,ltr_jan1,ltr_jan21,jan_net_rides,lifetime_stage_jan1,lifetime_stage_jan21,recency_segment_jan1,recency_segment_jan21,service_affinity_jan1,service_affinity_jan21,group_tc


In [198]:
df_test1[df_test1['user_id'].isin(df_train_control['user_id'])]

,user_id,city,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,time_to_cancel,fe2rr_session,g2n_session,fe2net_session,ltr_jan1,ltr_jan21,jan_net_rides,lifetime_stage_jan1,lifetime_stage_jan21,recency_segment_jan1,recency_segment_jan21,service_affinity_jan1,service_affinity_jan21,group_tc


In [199]:
df_test1[df_test1['user_id'].isin(df_test2['user_id'])]

,user_id,city,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,time_to_cancel,fe2rr_session,g2n_session,fe2net_session,ltr_jan1,ltr_jan21,jan_net_rides,lifetime_stage_jan1,lifetime_stage_jan21,recency_segment_jan1,recency_segment_jan21,service_affinity_jan1,service_affinity_jan21,group_tc


In [200]:
df_test2[df_test2['user_id'].isin(df_train_control['user_id'])]

,user_id,city,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,time_to_cancel,fe2rr_session,g2n_session,fe2net_session,ltr_jan1,ltr_jan21,jan_net_rides,lifetime_stage_jan1,lifetime_stage_jan21,recency_segment_jan1,recency_segment_jan21,service_affinity_jan1,service_affinity_jan21,group_tc


In [201]:
df_test2[df_test2['user_id'].isin(df_test1['user_id'])]

,user_id,city,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,time_to_cancel,fe2rr_session,g2n_session,fe2net_session,ltr_jan1,ltr_jan21,jan_net_rides,lifetime_stage_jan1,lifetime_stage_jan21,recency_segment_jan1,recency_segment_jan21,service_affinity_jan1,service_affinity_jan21,group_tc


## User Selector for Experiment

In [202]:
df_train_control

,user_id,city,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,time_to_cancel,fe2rr_session,g2n_session,fe2net_session,ltr_jan1,ltr_jan21,jan_net_rides,lifetime_stage_jan1,lifetime_stage_jan21,recency_segment_jan1,recency_segment_jan21,service_affinity_jan1,service_affinity_jan21,group_tc
685027,5eddb5869adf085bf4f6a237,Hyderabad,MALE,9.908611e+09,30,3,2,10.00,66.67,6.67,7,2,2,0,0,0,1,6.0,73,0,28.57,100.00,28.57,10.0,12.0,2.0,HOOK,HOOK,RECENT,STATIONARY,ONLY_LINK,ONLY_LINK,control
864266,59a3f951f2b6708059499182,Bangalore,MALE,9.964336e+09,7,4,3,57.14,75.00,42.86,4,3,3,0,1,0,0,51.0,2240,0,75.00,100.00,75.00,67.0,74.0,7.0,COMMITTED,CHURN_OTB,RECENT,STATIONARY,ONLY_LINK,ONLY_LINK,control
1962366,62ea23dbcc3711c71c9144d2,Bangalore,FEMALE,6.364831e+09,4,1,1,25.00,100.00,25.00,2,1,1,0,0,0,0,20.0,12,0,50.00,100.00,50.00,0.0,1.0,1.0,UNKNOWN,HANDHOLDING,UNKNOWN,STATIONARY,UNKNOWN,ONLY_LINK,control
2592507,5cc68d861acc4f1c5503e8f4,Hyderabad,FEMALE,9.032534e+09,2,1,0,50.00,0.00,0.00,1,1,0,0,1,0,0,0.0,0,0,100.00,0.00,0.00,78.0,78.0,0.0,SOFT_CHURN,DORMANT,STATIONARY,DORMANT,BOTH,ONLY_LINK,control
3984576,627e277adc46f329781d3ae7,Bangalore,MALE,7.760409e+09,3,1,0,33.33,0.00,0.00,3,1,0,0,0,1,0,1.0,459,1030,33.33,0.00,0.00,4.0,4.0,0.0,INACTIVE,INACTIVE,INACTIVE,INACTIVE,UNKNOWN,ONLY_AUTO,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1161872,5f425a75979c25c2e4037078,Hyderabad,MALE,8.297340e+09,1,1,1,100.00,100.00,100.00,1,1,1,0,0,0,0,20.0,40,0,100.00,100.00,100.00,7.0,8.0,1.0,DORMANT,SUSTENANCE,DORMANT,RECENT,BOTH,BOTH,control
3099633,63386547abf21013a82acbfe,Bangalore,MALE,7.021108e+09,6,5,1,83.33,20.00,16.67,2,2,1,0,1,3,0,42.0,490,1367,100.00,50.00,50.00,6.0,9.0,3.0,HOOK,SUSTENANCE,RECENT,STATIONARY,ONLY_LINK,ONLY_LINK,control
3579345,5d8b0be4f354c15d38cbca76,Hyderabad,FEMALE,7.675974e+09,29,12,6,41.38,50.00,20.69,10,9,6,0,4,2,0,84.0,1105,650,90.00,66.67,60.00,205.0,213.0,8.0,COMMITTED,COMMITTED,RECENT,RECENT,ONLY_LINK,ONLY_LINK,control
1275283,62d57ab20fcf304e729b1cd4,Bangalore,FEMALE,8.724863e+09,37,22,3,59.46,13.64,8.11,10,10,3,0,11,3,5,62.0,1954,554,100.00,30.00,30.00,129.0,133.0,4.0,COMMITTED,COMMITTED,STATIONARY,RECENT,ONLY_AUTO,ONLY_AUTO,control


In [203]:
df_test1

,user_id,city,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,time_to_cancel,fe2rr_session,g2n_session,fe2net_session,ltr_jan1,ltr_jan21,jan_net_rides,lifetime_stage_jan1,lifetime_stage_jan21,recency_segment_jan1,recency_segment_jan21,service_affinity_jan1,service_affinity_jan21,group_tc
1426330,604b4dc6cc0b48ab45bb700f,Kolkata,FEMALE,7.865982e+09,46,23,18,50.00,78.26,39.13,21,20,18,0,3,1,1,175.0,7826,1193,95.24,90.0,85.71,152.0,177.0,25.0,COMMITTED,COMMITTED,RECENT,RECENT,ONLY_LINK,ONLY_LINK,test1
2398304,64776682ec2e94abe4bed627,Delhi,MALE,9.569475e+09,4,1,1,25.00,100.00,25.00,1,1,1,0,0,0,0,3.0,43,0,100.00,100.0,100.00,17.0,18.0,1.0,DORMANT,HOOK,DORMANT,RECENT,ONLY_LINK,ONLY_LINK,test1
1615858,63705237b48cdd1641f9b242,Bangalore,MALE,9.066857e+09,3,2,1,66.67,50.00,33.33,2,2,1,0,0,1,0,5.0,357,388,100.00,50.0,50.00,58.0,61.0,3.0,SUSTENANCE,COMMITTED,RECENT,STATIONARY,ONLY_LINK,ONLY_LINK,test1
267213,62bd2062ba38cb0b5e7211fe,Delhi,FEMALE,8.826052e+09,16,4,4,25.00,100.00,25.00,8,4,4,0,0,0,0,35.0,124,0,50.00,100.0,50.00,248.0,255.0,7.0,COMMITTED,COMMITTED,RECENT,STATIONARY,ONLY_AUTO,ONLY_AUTO,test1
733521,627bc9b787b0a686ddc5ff73,Jaipur,MALE,8.107491e+09,1,1,1,100.00,100.00,100.00,1,1,1,0,0,0,0,8.0,40,0,100.00,100.0,100.00,12.0,13.0,1.0,INACTIVE,HOOK,INACTIVE,RECENT,UNKNOWN,ONLY_LINK,test1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3925413,640b222c6a096750034049d6,Delhi,MALE,8.423494e+09,14,5,1,35.71,20.00,7.14,6,2,1,0,3,0,1,28.0,48,0,33.33,50.0,16.67,12.0,13.0,1.0,DORMANT,CHURN_OTB,DORMANT,RECENT,BOTH,ONLY_AUTO,test1
1159593,6503f570b3b19fa7e0e7009f,Delhi,MALE,9.792444e+09,6,1,0,16.67,0.00,0.00,5,1,0,0,1,0,0,2.0,0,0,20.00,0.0,0.00,1.0,1.0,0.0,DORMANT,INACTIVE,DORMANT,INACTIVE,ONLY_LINK,UNKNOWN,test1
2273153,5e0b9f3339136622c07819aa,Delhi,MALE,9.899410e+09,2,1,1,50.00,100.00,50.00,2,1,1,0,0,0,0,16.0,256,0,50.00,100.0,50.00,99.0,100.0,1.0,HOOK,HOOK,RECENT,RECENT,ONLY_LINK,ONLY_LINK,test1
2603675,642e812734270d4a0955d985,Delhi,MALE,7.807858e+09,5,1,0,20.00,0.00,0.00,2,1,0,0,0,1,0,3.0,30,225,50.00,0.0,0.00,41.0,41.0,0.0,SUSTENANCE,DORMANT,STATIONARY,DORMANT,ONLY_LINK,ONLY_LINK,test1


In [204]:
df_test2

,user_id,city,gender,customer_mobile,fe_count,rr_count,net_count,fe2rr,g2n,fe2net,fe_session,gross_sessions,net_sessions,cobrm,cobra,ocara,expired,eta,patience_time,time_to_cancel,fe2rr_session,g2n_session,fe2net_session,ltr_jan1,ltr_jan21,jan_net_rides,lifetime_stage_jan1,lifetime_stage_jan21,recency_segment_jan1,recency_segment_jan21,service_affinity_jan1,service_affinity_jan21,group_tc
1382355,6276436c30710835a8dd171b,Hyderabad,MALE,7.793930e+09,4,1,0,25.00,0.00,0.00,2,1,0,0,0,1,0,1.0,10,613,50.00,0.0,0.00,4.0,4.0,0.0,HANDHOLDING,DORMANT,STATIONARY,DORMANT,ONLY_LINK,ONLY_LINK,test2
2266824,5d4eb40455fbf50d45ed6a37,Jaipur,MALE,8.107344e+09,6,1,0,16.67,0.00,0.00,2,1,0,0,0,1,0,1.0,532,1231,50.00,0.0,0.00,113.0,114.0,1.0,SUSTENANCE,HOOK,RECENT,STATIONARY,ONLY_LINK,ONLY_LINK,test2
1690116,64fc8450a3be1d399566de0a,Bangalore,FEMALE,6.362761e+09,1,1,0,100.00,0.00,0.00,1,1,0,0,0,0,1,0.0,0,0,100.00,0.0,0.00,4.0,4.0,0.0,HANDHOLDING,HANDHOLDING,RECENT,STATIONARY,ONLY_AUTO,ONLY_LINK,test2
2486785,656ff880d04d825c1d030171,Delhi,MALE,8.510902e+09,6,1,1,16.67,100.00,16.67,3,1,1,0,0,0,0,8.0,24,0,33.33,100.0,33.33,1.0,2.0,1.0,DORMANT,HANDHOLDING,DORMANT,RECENT,ONLY_AUTO,ONLY_AUTO,test2
2589727,6423f295d7b7f17a14bb223d,Delhi,MALE,7.289088e+09,15,3,2,20.00,66.67,13.33,10,2,2,0,1,0,0,16.0,132,0,20.00,100.0,20.00,3.0,5.0,2.0,DORMANT,SUSTENANCE,DORMANT,RECENT,ONLY_LINK,ONLY_LINK,test2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2948862,5db724bec54ee1212528e56d,Hyderabad,MALE,7.730004e+09,3,2,2,66.67,100.00,66.67,3,2,2,0,0,0,0,36.0,272,0,66.67,100.0,66.67,547.0,549.0,2.0,DORMANT,SUSTENANCE,DORMANT,STATIONARY,ONLY_LINK,ONLY_LINK,test2
2819954,5e252a7b48bf6e0289861ef2,Delhi,MALE,7.906961e+09,10,3,1,30.00,33.33,10.00,4,1,1,1,0,0,1,26.0,24,0,25.00,100.0,25.00,9.0,10.0,1.0,DORMANT,HOOK,DORMANT,RECENT,ONLY_LINK,ONLY_LINK,test2
1480452,62ef8b7faa737791851e006b,Jaipur,FEMALE,9.100134e+09,7,1,0,14.29,0.00,0.00,4,1,0,0,0,1,0,6.0,73,135,25.00,0.0,0.00,25.0,25.0,0.0,DORMANT,DORMANT,DORMANT,DORMANT,ONLY_AUTO,ONLY_AUTO,test2
2930040,647ab29e0b7f5804c18392ea,Bangalore,MALE,8.147345e+09,8,3,0,37.50,0.00,0.00,3,1,0,0,0,3,0,11.0,32,1497,33.33,0.0,0.00,18.0,18.0,0.0,HOOK,DORMANT,STATIONARY,DORMANT,UNKNOWN,UNKNOWN,test2


In [205]:
df_train_control['customer_mobile'] = df_train_control['customer_mobile'].astype(float).astype(int)
df_train_control[['customer_mobile']].to_csv(local_datasets + 'processed/Ads_at_Captain_Dispatch_CONTROL_A.csv', 
                                             header=False,
                                             index=False )
# ADS_AT_CAPTAIN_DISPATCH_CONTROL_A

In [206]:
df_test1['customer_mobile'] = df_test1['customer_mobile'].astype(float).astype(int)
df_test1[['customer_mobile']].to_csv(local_datasets + 'processed/Ads_at_Captain_Dispatch_TEST_B.csv', 
                                     header=False,
                                     index=False )

In [207]:
df_test2['customer_mobile'] = df_test2['customer_mobile'].astype(float).astype(int)
df_test2[['customer_mobile']].to_csv(local_datasets + 'processed/Ads_at_Captain_Dispatch_TEST_C.csv', 
                                     header=False,
                                     index=False )